In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_dataset.csv")

df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount_(usd),location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [4]:
df["avg_order_value"] = df["purchase_amount_(usd)"] / df["previous_purchases"].replace(0, 1)

In [5]:
df["promo_dependency_score"] = df["discount_applied"] / df["purchase_amount_(usd)"].replace(0, 1)

TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [6]:
df = df.rename(columns={
    "purchase_amount_(usd)": "purchase_amount"
})

df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,avg_order_value
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,3.785714
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly,32.000000
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,3.173913
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,1.836735
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,1.580645


In [7]:
df["discount_applied_flag"] = df["discount_applied"].map({
    "Yes": 1,
    "No": 0
})

df["promo_code_used_flag"] = df["promo_code_used"].map({
    "Yes": 1,
    "No": 0
})

df["subscription_flag"] = df["subscription_status"].map({
    "Yes": 1,
    "No": 0
})

In [8]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,avg_order_value,discount_applied_flag,promo_code_used_flag,subscription_flag
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Express,Yes,Yes,14,Venmo,Fortnightly,3.785714,1,1,1
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Express,Yes,Yes,2,Cash,Fortnightly,32.000000,1,1,1
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Free Shipping,Yes,Yes,23,Credit Card,Weekly,3.173913,1,1,1
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Next Day Air,Yes,Yes,49,PayPal,Weekly,1.836735,1,1,1
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Free Shipping,Yes,Yes,31,PayPal,Annually,1.580645,1,1,1


In [11]:
frequency_map = {
    "Weekly": 7,
    "Fortnightly": 6,
    "Monthly": 5,
    "Quarterly": 4,
    "Every 3 Months": 4,
    "Annually": 2,
    "Bi-Weekly":1
}

df["frequency_score"] = df["frequency_of_purchases"].map(frequency_map)

In [12]:
df[df["frequency_score"].isnull()]["frequency_of_purchases"].unique()

array([], dtype=object)

In [13]:
df["satisfaction_flag"] = np.where(df["review_rating"] >= 4.0, 1, 0)

In [14]:
def satisfaction_level(rating):
    if rating >= 4.0:
        return "High Satisfaction"
    elif rating >= 3.0:
        return "Medium Satisfaction"
    else:
        return "Low Satisfaction"

df["satisfaction_level"] = df["review_rating"].apply(satisfaction_level)

In [15]:
df["promo_dependency_score"] = (
    df["discount_applied_flag"] + df["promo_code_used_flag"]
) / 2

In [16]:
def promo_dependency_level(score):
    if score == 1:
        return "High Promo Dependency"
    elif score == 0.5:
        return "Medium Promo Dependency"
    else:
        return "Low Promo Dependency"

df["promo_dependency_level"] = df["promo_dependency_score"].apply(promo_dependency_level)

In [17]:
df["value_score"] = (
    0.30 * df["purchase_amount"].rank(pct=True) +
    0.25 * df["previous_purchases"].rank(pct=True) +
    0.20 * df["frequency_score"].rank(pct=True) +
    0.10 * df["subscription_flag"] +
    0.10 * df["satisfaction_flag"] -
    0.05 * df["promo_dependency_score"]
)

In [18]:
df["value_tier"] = pd.qcut(
    df["value_score"],
    q=4,
    labels=["Low Value", "Mid Value", "High Value", "Premium Value"]
)

In [19]:
df["value_tier"].value_counts()

value_tier
Low Value        975
Mid Value        975
High Value       975
Premium Value    975
Name: count, dtype: int64

In [20]:
purchase_threshold = df["previous_purchases"].quantile(0.75)
amount_threshold = df["purchase_amount"].median()
frequency_threshold = df["frequency_score"].median()

df["loyalty_def_1"] = np.where(
    (df["previous_purchases"] >= purchase_threshold) &
    (df["purchase_amount"] >= amount_threshold) &
    (df["frequency_score"] >= frequency_threshold),
    1,
    0
)

In [21]:
df["loyalty_def_2"] = np.where(
    (df["previous_purchases"] >= purchase_threshold) &
    (df["frequency_score"] >= frequency_threshold) &
    (df["satisfaction_flag"] == 1) &
    (df["promo_dependency_score"] <= 0.5),
    1,
    0
)

In [22]:
loyalty_comparison = df.groupby("loyalty_def_1").agg({
    "purchase_amount": "mean",
    "previous_purchases": "mean",
    "frequency_score": "mean",
    "review_rating": "mean",
    "promo_dependency_score": "mean",
    "subscription_flag": "mean"
})

loyalty_comparison

,purchase_amount,previous_purchases,frequency_score,review_rating,promo_dependency_score,subscription_flag
loyalty_def_1,,,,,,
0,57.540380,23.376594,4.020969,3.745027,0.429017,0.270332
1,80.919137,44.137466,5.064690,3.802965,0.439353,0.266846


In [23]:
loyalty_comparison_2 = df.groupby("loyalty_def_2").agg({
    "purchase_amount": "mean",
    "previous_purchases": "mean",
    "frequency_score": "mean",
    "review_rating": "mean",
    "promo_dependency_score": "mean",
    "subscription_flag": "mean"
})

loyalty_comparison_2

,purchase_amount,previous_purchases,frequency_score,review_rating,promo_dependency_score,subscription_flag
loyalty_def_2,,,,,,
0,59.691689,24.499732,4.076944,3.717668,0.449598,0.282306
1,61.358824,44.041176,5.070588,4.471765,0.000000,0.000000


In [24]:
def create_customer_segment(row):
    if row["loyalty_def_2"] == 1 and row["value_tier"] in ["High Value", "Premium Value"]:
        return "Champions"
    
    elif row["previous_purchases"] >= purchase_threshold and row["promo_dependency_score"] >= 0.5:
        return "Promo-Dependent Regulars"
    
    elif row["previous_purchases"] < purchase_threshold and row["promo_dependency_score"] >= 0.5:
        return "One-Time Bargain Hunters"
    
    elif row["satisfaction_flag"] == 0 and row["previous_purchases"] < purchase_threshold:
        return "At-Risk Low Value Customers"
    
    else:
        return "Stable Mid-Value Customers"

df["customer_segment"] = df.apply(create_customer_segment, axis=1)

In [25]:
df["customer_segment"].value_counts()

customer_segment
One-Time Bargain Hunters       1235
Stable Mid-Value Customers     1087
At-Risk Low Value Customers     972
Promo-Dependent Regulars        442
Champions                       164
Name: count, dtype: int64

In [26]:
def age_group(age):
    if age < 25:
        return "18-24"
    elif age < 35:
        return "25-34"
    elif age < 45:
        return "35-44"
    elif age < 55:
        return "45-54"
    else:
        return "55+"

df["age_group"] = df["age"].apply(age_group)

In [27]:
df["spend_level"] = pd.qcut(
    df["purchase_amount"],
    q=3,
    labels=["Low Spend", "Medium Spend", "High Spend"]
)

In [28]:
df["retention_score"] = (
    0.6 * df["previous_purchases"].rank(pct=True) +
    0.4 * df["frequency_score"].rank(pct=True)
)

In [29]:
df["retention_level"] = pd.qcut(
    df["retention_score"],
    q=3,
    labels=["Low Retention", "Medium Retention", "High Retention"]
)

In [30]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,promo_dependency_level,value_score,value_tier,loyalty_def_1,loyalty_def_2,customer_segment,age_group,spend_level,retention_score,retention_level
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,High Promo Dependency,0.403006,Mid Value,0,0,One-Time Bargain Hunters,55+,Medium Spend,0.480128,Medium Retention
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,High Promo Dependency,0.381378,Mid Value,0,0,One-Time Bargain Hunters,18-24,Medium Spend,0.335359,Low Retention
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,High Promo Dependency,0.547269,Premium Value,0,0,One-Time Bargain Hunters,45-54,Medium Spend,0.643026,High Retention
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,High Promo Dependency,0.739019,Premium Value,1,0,Promo-Dependent Regulars,18-24,High Spend,0.956179,High Retention
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,High Promo Dependency,0.358051,Mid Value,0,0,One-Time Bargain Hunters,45-54,Medium Spend,0.454718,Medium Retention


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   customer_id             3900 non-null   int64   
 1   age                     3900 non-null   int64   
 2   gender                  3900 non-null   object  
 3   item_purchased          3900 non-null   object  
 4   category                3900 non-null   object  
 5   purchase_amount         3900 non-null   int64   
 6   location                3900 non-null   object  
 7   size                    3900 non-null   object  
 8   color                   3900 non-null   object  
 9   season                  3900 non-null   object  
 10  review_rating           3900 non-null   float64 
 11  subscription_status     3900 non-null   object  
 12  shipping_type           3900 non-null   object  
 13  discount_applied        3900 non-null   object  
 14  promo_code_used         

In [32]:
df.isnull().sum()

customer_id               0
age                       0
gender                    0
item_purchased            0
category                  0
purchase_amount           0
location                  0
size                      0
color                     0
season                    0
review_rating             0
subscription_status       0
shipping_type             0
discount_applied          0
promo_code_used           0
previous_purchases        0
payment_method            0
frequency_of_purchases    0
avg_order_value           0
discount_applied_flag     0
promo_code_used_flag      0
subscription_flag         0
frequency_score           0
satisfaction_flag         0
satisfaction_level        0
promo_dependency_score    0
promo_dependency_level    0
value_score               0
value_tier                0
loyalty_def_1             0
loyalty_def_2             0
customer_segment          0
age_group                 0
spend_level               0
retention_score           0
retention_level     

In [33]:
df[[
    "purchase_amount",
    "previous_purchases",
    "frequency_score",
    "promo_dependency_score",
    "satisfaction_flag",
    "value_score",
    "value_tier",
    "loyalty_def_1",
    "loyalty_def_2",
    "customer_segment",
    "age_group",
    "spend_level",
    "retention_score",
    "retention_level"
]].head()

,purchase_amount,previous_purchases,frequency_score,promo_dependency_score,satisfaction_flag,value_score,value_tier,loyalty_def_1,loyalty_def_2,customer_segment,age_group,spend_level,retention_score,retention_level
0,53,14,6,1.0,0,0.403006,Mid Value,0,0,One-Time Bargain Hunters,55+,Medium Spend,0.480128,Medium Retention
1,64,2,6,1.0,0,0.381378,Mid Value,0,0,One-Time Bargain Hunters,18-24,Medium Spend,0.335359,Low Retention
2,73,23,7,1.0,0,0.547269,Premium Value,0,0,One-Time Bargain Hunters,45-54,Medium Spend,0.643026,High Retention
3,90,49,7,1.0,0,0.739019,Premium Value,1,0,Promo-Dependent Regulars,18-24,High Spend,0.956179,High Retention
4,49,31,2,1.0,0,0.358051,Mid Value,0,0,One-Time Bargain Hunters,45-54,Medium Spend,0.454718,Medium Retention


In [34]:
df.to_csv("../data/processed/customer_features.csv", index=False)